# Check research mathematics with NumPy

Run locally or in Colab. Each task turns a mathematical claim into a numerical check. Keep random
seeds fixed for reproducibility, then change seeds or initial conditions to see which conclusions
are robust.


In [ ]:
import numpy as np

X = np.array([[1.0, 2.0], [0.0, -1.0], [2.0, 1.0]])
y = np.array([1.0, 2.0, 0.0])

def least_squares(w):
    residual = X @ w - y
    return 0.5 * residual @ residual


## 1. Check a vector gradient

Derive the gradient of the supplied half squared-error objective. Implement it and check it with
central finite differences along every coordinate.


In [ ]:
def gradient(w):
    # TODO: return the analytic gradient.
    raise NotImplementedError

def finite_difference(f, w, step=1e-5):
    # TODO: perturb one coordinate at a time.
    raise NotImplementedError


In [ ]:
for w in [np.zeros(2), np.array([0.7, -0.4])]:
    assert np.allclose(gradient(w), finite_difference(least_squares, w), atol=1e-8)
assert np.allclose(gradient(np.zeros(2)), [-1, 0])
print("OK")


## 2. Take a Newton step

Compute the least-squares Hessian and solve the Newton linear system. Use a solver instead of
forming an inverse. Explain why a single step reaches the minimum for this full-rank quadratic.


In [ ]:
def newton_step(w):
    # TODO: solve H delta = gradient(w) and update w.
    raise NotImplementedError


In [ ]:
expected = np.linalg.lstsq(X, y, rcond=None)[0]
for w in [np.array([9.0, -3.0]), np.zeros(2)]:
    assert np.allclose(newton_step(w), expected)
    assert np.linalg.norm(gradient(newton_step(w))) < 1e-10
print("OK")


## 3. Check constrained optimality

Minimise half the squared distance from a vector v subject to x being elementwise nonnegative.
Use the convention g(x) = -x <= 0, so stationarity is x - v - lambda = 0. Derive and return x and
lambda, then check feasibility, stationarity and complementary slackness.


In [ ]:
def project_nonnegative(v):
    # TODO: return the primal point and its nonnegative multipliers.
    raise NotImplementedError


In [ ]:
for v in [np.array([-3.0, 2.0, 0.0]), np.array([1.0, -2.0])]:
    point, multipliers = project_nonnegative(v)
    assert np.all(point >= 0) and np.all(multipliers >= 0)
    assert np.allclose(point - v - multipliers, 0)
    assert np.allclose(point * multipliers, 0)
print("OK")


## 4. Report a Monte Carlo estimate

Simulate a correlated Gaussian by multiplying independent standard normal draws by a Cholesky
factor. Implement `sample_gaussian(covariance, n, seed)`: return an n×2 array
`default_rng(seed).standard_normal((n, 2)) @ L.T`, where L is the Cholesky factor of the covariance.
Estimate the variance of the first coordinate as the mean of its squared draws.
Report the sample standard deviation of those squares divided by the square root of the draw count.


In [ ]:
def sample_gaussian(covariance, n, seed):
    # TODO: return n correlated draws using the Cholesky factor of covariance.
    raise NotImplementedError


def estimate_variance(covariance, n, seed):
    # TODO: sample with sample_gaussian, square coordinate zero, and return the mean and standard error.
    raise NotImplementedError


In [ ]:
covariance = np.array([[2.0, 0.5], [0.5, 1.0]])
estimate, se = estimate_variance(covariance, 20000, 14)
assert abs(estimate - 2.0) < 4 * se
assert 0.015 < se < 0.03
_, smaller_se = estimate_variance(covariance, 80000, 14)
assert 0.4 < smaller_se / se < 0.6
assert np.allclose(np.cov(sample_gaussian(covariance, 20000, 14), rowvar=False), covariance, atol=0.06)
print(f"Variance {estimate:.3f} with Monte Carlo standard error {se:.3f}")
print("OK")


## 5. Check the ELBO identity by enumeration

Use a discrete model with strictly positive probabilities. Given a vector of joint probabilities
p(x,z) for a fixed observation and a trial distribution q(z), compute log evidence, the ELBO and
KL divergence from q to the exact posterior. Use natural logarithms throughout.


In [ ]:
def elbo_terms(joint, q):
    # TODO: normalise the posterior and compute the three scalars.
    raise NotImplementedError


In [ ]:
joint = np.array([0.1, 0.3])
exact, bound, gap = elbo_terms(joint, np.array([0.5, 0.5]))
assert np.isclose(exact, np.log(0.4))
assert gap > 0 and np.isclose(exact, bound + gap)
exact, bound, gap = elbo_terms(joint, np.array([0.25, 0.75]))
assert np.isclose(exact, bound) and np.isclose(gap, 0)
print("OK")


## 6. Write a derivation note

**Your answer:** Annotate the ELBO identity line by line. Label the step at which you use Bayes'
rule, give the shape and domain of every object, and state when the lower bound becomes tight.
Explain what the numerical check establishes and give one assumption it cannot verify for a
continuous latent-variable model.
